In [ ]:
import pandas as pd

BASE_URL = "https://azure.abhi.dedyn.io/iot/"
OUTPUT_CSV = "final_labeled_data.csv"

# Desired column headers
COLUMNS = [
    "temperature",
    "pressure",
    "humidity",
    "gas",
    "aqi",
    "time_stamp_esp",
    "time_stamp_server"
]

all_data = []
file_index = 1

while True:
    try:
        file_url = f"{BASE_URL}csv_data_{file_index}.csv"
        print(f"Fetching: {file_url}")

        # Read CSV without header
        df = pd.read_csv(file_url, header=None)

        # Validate column count
        if df.shape[1] != len(COLUMNS):
            print(f"Column mismatch in csv_data_{file_index}.csv — stopping.")
            break

        # Assign headers
        df.columns = COLUMNS

        all_data.append(df)
        file_index += 1

    except Exception as e:
        print(f"No more files found. Stopping at csv_data_{file_index - 1}.csv")
        break

# Combine all CSVs
final_df = pd.concat(all_data, ignore_index=True)

# Save final CSV
final_df.to_csv(OUTPUT_CSV, index=False)

print(f"\n✅ Final CSV saved as: {OUTPUT_CSV}")
print(f"Total rows collected: {len(final_df)}")


Fetching: https://azure.abhi.dedyn.io/iot/csv_data_1.csv
Fetching: https://azure.abhi.dedyn.io/iot/csv_data_2.csv
Fetching: https://azure.abhi.dedyn.io/iot/csv_data_3.csv
Fetching: https://azure.abhi.dedyn.io/iot/csv_data_4.csv
Fetching: https://azure.abhi.dedyn.io/iot/csv_data_5.csv
Fetching: https://azure.abhi.dedyn.io/iot/csv_data_6.csv
Fetching: https://azure.abhi.dedyn.io/iot/csv_data_7.csv
Fetching: https://azure.abhi.dedyn.io/iot/csv_data_8.csv
Fetching: https://azure.abhi.dedyn.io/iot/csv_data_9.csv
Fetching: https://azure.abhi.dedyn.io/iot/csv_data_10.csv
Fetching: https://azure.abhi.dedyn.io/iot/csv_data_11.csv
Fetching: https://azure.abhi.dedyn.io/iot/csv_data_12.csv
No more files found. Stopping at csv_data_11.csv

✅ Final CSV saved as: final_labeled_data.csv
Total rows collected: 20746


In [ ]:
# import pandas as pd

# # Path to your original CSV file
# input_csv = "csv_data.csv"     # change this to your actual file name
# output_csv = "labeled_data.csv"

# # Define the headers you want
# columns = [
#     "temperature",
#     "pressure",
#     "humidity",
#     "gas",
#     "aqi",
#     "time_stamp_esp",
#     "time_stamp_server"
# ]

# # Read CSV WITHOUT header
# df = pd.read_csv(input_csv, header=None)

# # Assign column names
# df.columns = columns

# # Save the new CSV
# df.to_csv(output_csv, index=False)

# print("New CSV saved with headers successfully!")


New CSV saved with headers successfully!


🧩 CELL 1 — Load, Sort, and Prepare Base Features

In [ ]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv("final_labeled_data.csv")

# Sort by time (CRITICAL for time series)
df = df.sort_values("time_stamp_esp").reset_index(drop=True)

print("Data loaded & sorted")
print(df.head(3))
print("Rows:", len(df))


Data loaded & sorted
   temperature  pressure  humidity  gas    aqi  time_stamp_esp  \
0        36.12      1013     24.95   73  39187           61.67   
1        36.25      1013     25.22   73  39245           61.91   
2        36.56      1013     25.19   68  39405           61.91   

                time_stamp_server  
0  2026-01-27T04:35:10.136277672Z  
1  2026-01-27T04:36:07.580162373Z  
2  2026-01-27T04:38:47.733344115Z  
Rows: 20746


🧩 CELL 2 — Pollution Aggregation (Gas + AQI)

In [ ]:
# Percentiles for normalization
gas_q80 = df['gas'].quantile(0.8)
aqi_q80 = df['aqi'].quantile(0.8)

# Normalize
df['gas_norm'] = df['gas'] / gas_q80
df['aqi_norm'] = df['aqi'] / aqi_q80

# Aggregated pollution index
df['pollution_index'] = (
    0.6 * df['gas_norm'] +
    0.4 * df['aqi_norm']
)

print(df[['gas_norm','aqi_norm','pollution_index']].describe())


           gas_norm      aqi_norm  pollution_index
count  20746.000000  20746.000000     20746.000000
mean       0.934557      0.667389         0.827690
std        0.096415      0.324509         0.143222
min        0.392857      0.008468         0.246457
25%        0.892857      0.400876         0.759473
50%        0.976190      0.673448         0.866856
75%        1.000000      0.945135         0.937704
max        1.047619      1.227015         1.009517


🧩 CELL 3 — Logical Labeling (Safe & Deterministic)

In [ ]:
hum_q20  = df['humidity'].quantile(0.2)
hum_q80  = df['humidity'].quantile(0.8)
temp_q20 = df['temperature'].quantile(0.2)
temp_q80 = df['temperature'].quantile(0.8)

def assign_label(row, margin=0.15):
    scores = {
        "polluted": row["pollution_index"],
        "humid": row["humidity"] / hum_q80,
        "dry": hum_q20 / max(row["humidity"], 1),
        "hot": row["temperature"] / temp_q80,
        "cold": temp_q20 / max(row["temperature"], 1),
    }

    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    top, top_val = sorted_scores[0]
    second, sec_val = sorted_scores[1]

    if top_val < 1.0:
        return "comfortable"
    if abs(top_val - sec_val) < margin:
        return "uncomfortable"
    if top == "polluted":
        return "polluted"
    return "uncomfortable"

df["label"] = df.apply(assign_label, axis=1)

print(df["label"].value_counts())


label
comfortable      12301
uncomfortable     8445
Name: count, dtype: int64


🧩 CELL 4 — Reduce Labels (TinyML-Optimal)

In [ ]:
def reduce_label(lbl):
    if lbl == "polluted":
        return "polluted"
    if lbl == "comfortable":
        return "comfortable"
    return "uncomfortable"

df["label_final"] = df["label"].apply(reduce_label)
df.to_csv("row_level_labeled_timeseries.csv", index=False)

print("✅ Saved: row_level_labeled_timeseries.csv")
print("Columns:")
print(df.columns.tolist())
print("\nSample rows:")
print(df.head(5))

print("Final labels:")
print(df["label_final"].value_counts())


✅ Saved: row_level_labeled_timeseries.csv
Columns:
['temperature', 'pressure', 'humidity', 'gas', 'aqi', 'time_stamp_esp', 'time_stamp_server', 'gas_norm', 'aqi_norm', 'pollution_index', 'label', 'label_final']

Sample rows:
   temperature  pressure  humidity  gas    aqi  time_stamp_esp  \
0        36.12      1013     24.95   73  39187           61.67   
1        36.25      1013     25.22   73  39245           61.91   
2        36.56      1013     25.19   68  39405           61.91   
3        35.94      1013     25.24   74  39114           62.15   
4        36.12      1013     24.83   73  39203           62.15   

                time_stamp_server  gas_norm  aqi_norm  pollution_index  \
0  2026-01-27T04:35:10.136277672Z  0.869048  1.156232         0.983921   
1  2026-01-27T04:36:07.580162373Z  0.869048  1.157943         0.984606   
2  2026-01-27T04:38:47.733344115Z  0.809524  1.162664         0.950780   
3  2026-01-27T04:33:56.820514411Z  0.880952  1.154078         0.990202   
4  2026-

🧩 CELL 5 — Build Sliding Windows (Time Series Core)

In [ ]:
WINDOW_SIZE = 8
FEATURES = ['temperature','humidity','pressure','pollution_index']

# Scaling (MUST match Arduino)
def scale_row(row):
    return np.array([
        row[0] / 50.0,
        row[1] / 100.0,
        (row[2] - 1000.0) / 50.0,
        row[3] / 2.0
    ], dtype=np.float32)

X, y = [], []

for i in range(len(df) - WINDOW_SIZE):
    window = df.iloc[i:i+WINDOW_SIZE][FEATURES].values
    window_scaled = np.array([scale_row(r) for r in window])
    X.append(window_scaled)
    y.append(df.iloc[i+WINDOW_SIZE-1]["label_final"])

X = np.array(X)
y = np.array(y)

print("Windowed shape:", X.shape)
print("Labels:", np.unique(y))


Windowed shape: (20738, 8, 4)
Labels: ['comfortable' 'uncomfortable']


CELL 6 — Encode Labels & Train/Test Split

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc,
    test_size=0.2,
    stratify=y_enc,
    random_state=42
)

print("Classes:", le.classes_)
X_flat = X.reshape(X.shape[0], -1)

feature_names = []
for t in range(WINDOW_SIZE):
    feature_names += [
        f"temp_t-{WINDOW_SIZE-1-t}",
        f"hum_t-{WINDOW_SIZE-1-t}",
        f"press_t-{WINDOW_SIZE-1-t}",
        f"pollution_t-{WINDOW_SIZE-1-t}"
    ]

df_windows = pd.DataFrame(X_flat, columns=feature_names)
df_windows["label"] = le.inverse_transform(y_enc[:len(df_windows)])

df_windows.to_csv("windowed_timeseries_dataset.csv", index=False)

print("✅ Saved: windowed_timeseries_dataset.csv")
print("Shape:", df_windows.shape)
print("\nSample window:")
print(df_windows.head(2))


Classes: ['comfortable' 'uncomfortable']
✅ Saved: windowed_timeseries_dataset.csv
Shape: (20738, 33)

Sample window:
   temp_t-7  hum_t-7  press_t-7  pollution_t-7  temp_t-6  hum_t-6  press_t-6  \
0    0.7224   0.2495       0.26       0.491961    0.7250   0.2522       0.26   
1    0.7250   0.2522       0.26       0.492303    0.7312   0.2519       0.26   

   pollution_t-6  temp_t-5  hum_t-5  ...  pollution_t-2  temp_t-1  hum_t-1  \
0       0.492303    0.7312   0.2519  ...       0.502398    0.7906   0.2244   
1       0.475390    0.7188   0.2524  ...       0.479264    0.7900   0.2230   

   press_t-1  pollution_t-1  temp_t-0  hum_t-0  press_t-0  pollution_t-0  \
0       0.26       0.479264    0.7900   0.2230       0.26       0.482594   
1       0.26       0.482594    0.7302   0.2535       0.26       0.464611   

           label  
0  uncomfortable  
1  uncomfortable  

[2 rows x 33 columns]


🧩 CELL 7 — TinyML-Optimized Conv1D Model

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(WINDOW_SIZE, 4)),
    layers.Conv1D(8, kernel_size=3, activation='relu'),
    layers.Flatten(),
    layers.Dense(8, activation='relu'),
    layers.Dense(len(le.classes_), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 6, 8)           │           104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 48)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8)              │           392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │            18 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 514 (2.01 KB)

 Trainable params: 514 (2.01 KB)

 Non-trainable params: 0 (0.00 B)

🧩 CELL 8 — Train & Evaluate FLOAT Model

In [ ]:
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(class_weights))

model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=16,
    validation_split=0.1,
    class_weight=class_weights,
    verbose=1
)

y_pred = model.predict(X_test).argmax(axis=1)

print("FLOAT accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))


Epoch 1/50
934/934 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6131 - loss: 0.6083 - val_accuracy: 0.7071 - val_loss: 0.5070
Epoch 2/50
934/934 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7265 - loss: 0.5159 - val_accuracy: 0.7492 - val_loss: 0.5103
Epoch 3/50
934/934 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7531 - loss: 0.4960 - val_accuracy: 0.7553 - val_loss: 0.5088
Epoch 4/50
934/934 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.7499 - loss: 0.4922 - val_accuracy: 0.7498 - val_loss: 0.4646
Epoch 5/50
934/934 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.7596 - loss: 0.4804 - val_accuracy: 0.7215 - val_loss: 0.4600
Epoch 6/50
934/934 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7666 - loss: 0.4775 - val_accuracy: 0.7613 - val_loss: 0.4530
Epoch 7/50
934/934 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7682 - loss: 0.4686 - val_accuracy: 0.7715 - val_loss: 0.4520
Epoch 8/50
934/934 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7647 - loss: 0.4602 - val_accuracy: 0.

In [ ]:
import joblib

joblib.dump(le, "label_encoder.pkl")
print("✅ Saved label encoder: label_encoder.pkl")
print("Class order:", le.classes_)


✅ Saved label encoder: label_encoder.pkl
Class order: ['comfortable' 'uncomfortable']


In [ ]:
# Save trained FLOAT model (before INT8)
model.save("timeseries_conv1d_float.h5")

print("✅ Saved trained float model: timeseries_conv1d_float.h5")
print("Model input shape:", model.input_shape)
print("Model output shape:", model.output_shape)


✅ Saved trained float model: timeseries_conv1d_float.h5
Model input shape: (None, 8, 4)
Model output shape: (None, 2)


🧩 CELL 9 — INT8 Quantization

In [ ]:
def representative_dataset():
    for i in range(200):
        yield [X_train[i:i+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()

with open("timeseries_tinyml_int8.tflite", "wb") as f:
    f.write(tflite_model)

print("INT8 model size (bytes):", len(tflite_model))


Saved artifact at '/tmp/tmpbwf8jatk'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 8, 4), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  140583480146448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140583480147216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140583480147024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140583480147792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140583480145104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140583480145488: TensorSpec(shape=(), dtype=tf.resource, name=None)
INT8 model size (bytes): 4216


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


🧩 CELL 10 — Evaluate INT8 Model

In [ ]:
interpreter = tf.lite.Interpreter(model_path="timeseries_tinyml_int8.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

in_scale, in_zero = input_details[0]['quantization']
out_scale, out_zero = output_details[0]['quantization']

y_pred_int8 = []

for x in X_test:
    x_q = x / in_scale + in_zero
    x_q = np.clip(x_q, -128, 127).astype(np.int8)
    x_q = np.expand_dims(x_q, axis=0)

    interpreter.set_tensor(input_details[0]['index'], x_q)
    interpreter.invoke()

    out = interpreter.get_tensor(output_details[0]['index'])[0]
    out = (out.astype(np.float32) - out_zero) * out_scale
    y_pred_int8.append(np.argmax(out))

print("INT8 accuracy:", accuracy_score(y_test, y_pred_int8))
print(classification_report(y_test, y_pred_int8, target_names=le.classes_))


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


INT8 accuracy: 0.9831243972999035
               precision    recall  f1-score   support

  comfortable       1.00      0.98      0.99      2460
uncomfortable       0.96      0.99      0.98      1688

     accuracy                           0.98      4148
    macro avg       0.98      0.98      0.98      4148
 weighted avg       0.98      0.98      0.98      4148

